<a href="https://colab.research.google.com/github/HasiburRahman404/AgroLink/blob/main/nlp1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
print("Please provide more details on how you want to connect your Claude account.")
# The original content of this cell was:
# !pip install -q datasets faiss-gpu sentence-transformers bnlp_toolkit tqdm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.2/135.2 MB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.6/68.6 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 608.4/608.4 kB 42.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 73.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.6/2.6 MB 101.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 72.5 MB/s eta 0:00:00


In [2]:
from datasets import load_dataset

# Bengali Wikipedia dump, streaming avoids downloading the whole thing at once
wiki = load_dataset("wikimedia/wikipedia", "20231101.bn", split="train", streaming=True)

# Take a manageable subset for a course project — e.g. first 5000 articles
articles = []
for i, article in enumerate(wiki):
    if i >= 5000:
        break
    articles.append(article)

print(len(articles), "articles loaded")
print(articles[0]["title"])
print(articles[0]["text"][:500])

README.md:   0%|          | 0.00/131k [00:00<?, ?B/s]

5000 articles loaded
বাংলা ভাষা
বাংলা ভাষা (বাঙলা, বাঙ্গলা, তথা বাঙ্গালা নামেও পরিচিত) একটি ইন্দো-আর্য ভাষা, যা দক্ষিণ এশিয়ার বাঙালি জাতির প্রধান কথ্য ও লেখ্য ভাষা। মাতৃভাষীর সংখ্যায় বাংলা ইন্দো-ইউরোপীয় ভাষা পরিবারের পঞ্চম ও মোট ব্যবহারকারীর সংখ্যা অনুসারে বাংলা বিশ্বের ষষ্ঠ বৃহত্তম ভাষা। বাংলা সার্বভৌম ভাষাভিত্তিক জাতিরাষ্ট্র বাংলাদেশের একমাত্র রাষ্ট্রভাষা তথা সরকারি ভাষা এবং ভারতের পশ্চিমবঙ্গ, ত্রিপুরা, আসামের বরাক উপত্যকার সরকারি ভাষা। বঙ্গোপসাগরে অবস্থিত আন্দামান দ্বীপপুঞ্জের প্রধান কথ্য ভাষা বাংলা। এছাড়া ভারতের ঝাড়খণ


In [3]:
import re

def clean_text(text):
    text = re.sub(r'\n+', ' ', text)          # collapse newlines
    text = re.sub(r'\s+', ' ', text)           # collapse whitespace
    text = re.sub(r'\[\d+\]', '', text)        # remove citation markers like [1]
    return text.strip()

for a in articles:
    a["text"] = clean_text(a["text"])

In [4]:
def chunk_text(text, min_words=100, max_words=200):
    words = text.split()
    chunks = []
    for i in range(0, len(words), max_words):
        chunk = " ".join(words[i:i+max_words])
        if len(chunk.split()) >= min_words:  # drop tiny trailing chunks
            chunks.append(chunk)
    return chunks

passages = []  # each entry: {"text":..., "title":..., "article_id":...}
for idx, a in enumerate(articles):
    for c in chunk_text(a["text"]):
        passages.append({"text": c, "title": a["title"], "article_id": idx})

print(len(passages), "passages created")
print(passages[0])

19129 passages created
{'text': 'বাংলা ভাষা (বাঙলা, বাঙ্গলা, তথা বাঙ্গালা নামেও পরিচিত) একটি ইন্দো-আর্য ভাষা, যা দক্ষিণ এশিয়ার বাঙালি জাতির প্রধান কথ্য ও লেখ্য ভাষা। মাতৃভাষীর সংখ্যায় বাংলা ইন্দো-ইউরোপীয় ভাষা পরিবারের পঞ্চম ও মোট ব্যবহারকারীর সংখ্যা অনুসারে বাংলা বিশ্বের ষষ্ঠ বৃহত্তম ভাষা। বাংলা সার্বভৌম ভাষাভিত্তিক জাতিরাষ্ট্র বাংলাদেশের একমাত্র রাষ্ট্রভাষা তথা সরকারি ভাষা এবং ভারতের পশ্চিমবঙ্গ, ত্রিপুরা, আসামের বরাক উপত্যকার সরকারি ভাষা। বঙ্গোপসাগরে অবস্থিত আন্দামান দ্বীপপুঞ্জের প্রধান কথ্য ভাষা বাংলা। এছাড়া ভারতের ঝাড়খণ্ড, বিহার, মেঘালয়, মিজোরাম, ওড়িশা রাজ্যগুলোতে উল্লেখযোগ্য পরিমাণে বাংলাভাষী জনগণ রয়েছে। ২০১১ সালের আদমশুমারি অনুযায়ী, ভারতের মোট জনসংখ্যার ৮.০৩ শতাংশ মানুষ বাংলা ভাষায় কথা বলে এবং হিন্দির পরেই ভারতে সর্বাধিক প্রচলিত ভাষা - বাংলা। এছাড়াও মধ্য প্রাচ্য, আমেরিকা ও ইউরোপে উল্লেখযোগ্য পরিমাণে বাংলাভাষী অভিবাসী রয়েছে। সারা বিশ্বে সব মিলিয়ে ২৭.৬ কোটির অধিক লোক দৈনন্দিন জীবনে বাংলা ব্যবহার করে। বাংলাদেশের জাতীয় সঙ্গীত এবং ভারতের জাতীয় সঙ্গীত ও স্তোত্র বাংলাতে রচ

In [5]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("intfloat/multilingual-e5-base", device="cuda")

# e5 models require a "query: " or "passage: " prefix — this is specific to e5,
# it was trained this way so it knows which side of a search this text is on
passage_texts = ["passage: " + p["text"] for p in passages]

embeddings = model.encode(
    passage_texts,
    batch_size=64,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True  # important: lets us use cosine similarity via dot product
)

print(embeddings.shape)  # (num_passages, 768)

modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/179k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/694 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.11GB            

...

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/280 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/200 [00:00<?, ?B/s]

Batches:   0%|          | 0/299 [00:00<?, ?it/s]

(19129, 768)


In [6]:
import faiss
import numpy as np

dim = embeddings.shape[1]  # 768
index = faiss.IndexFlatIP(dim)  # IP = inner product = cosine similarity (since normalized)
index.add(embeddings.astype(np.float32))

print("Vectors in index:", index.ntotal)

Vectors in index: 19129


In [7]:
def search(query, k=3):
    query_vec = model.encode(["query: " + query], normalize_embeddings=True)
    scores, indices = index.search(query_vec.astype(np.float32), k)
    for score, idx in zip(scores[0], indices[0]):
        print(f"Score: {score:.3f} | Title: {passages[idx]['title']}")
        print(passages[idx]['text'][:200], "...\n")

search("বাংলাদেশের রাজধানী কোথায়?")  # "Where is the capital of Bangladesh?"

Score: 0.846 | Title: ঢাকা
প্রাদেশিক রাজধানী ছিল। মুঘল সম্রাট জাহাঙ্গীরের শাসনামলে এই শহর জাহাঙ্গীর নগর নামে পরিচিত ছিলো। বিশ্বব্যাপী মসলিন বাণিজ্যের একটি কেন্দ্র ছিলো ঢাকা এবং বিশ্বের বিভিন্ন স্থান থেকে ব্যবসায়ীগণ এখানে বাণিজ ...

Score: 0.844 | Title: ঢাকা
বাংলার রাজধানী হলেও সুবাহ বাংলার রাজধানী বারবার পরিবর্তন করা হয়েছে। ১৬৫০ খ্রিষ্টাব্দে সুবেদার শাহ সুজা রাজধানী আবার রাজমহলে স্থানান্তর করেছিলেন। শাহ সুজার পতনের পর ১৬৬০ খ্রিষ্টাব্দে সুবেদার মীর জুমলা ...

Score: 0.838 | Title: ঢাকা জেলা
ঢাকা জেলা বাংলাদেশের মধ্যাঞ্চলের ঢাকা বিভাগের একটি জেলা। বাংলাদেশের রাজধানী ঢাকা শহরটি এই জেলায় অবস্থিত। অবস্থানগত কারণে এটি বাংলাদেশের একটি বিশেষ শ্রেণীভুক্ত জেলা। ইতিহাস বাংলাদেশের রাজধানী ঢাকা মোঘ ...



In [11]:
import pickle

faiss.write_index(index, "wiki_bn.index")

with open("passages.pkl", "wb") as f:
    pickle.dump(passages, f)

print("Saved: wiki_bn.index, passages.pkl")

Saved: wiki_bn.index, passages.pkl


In [12]:
import os
print(os.listdir())

['.config', 'wiki_bn.index', 'passages.pkl', 'sample_data']


In [16]:
!pip install -q transformers datasets evaluate accelerate

In [18]:
from huggingface_hub import list_repo_files
files = list_repo_files("csebuetnlp/squad_bn", repo_type="dataset")
print(files)

['.gitattributes', 'README.md', 'data/squad_bn.tar.bz2', 'squad_bn.py']


In [19]:
from huggingface_hub import list_repo_files
files = list_repo_files("csebuetnlp/squad_bn", repo_type="dataset")
print(files)

['.gitattributes', 'README.md', 'data/squad_bn.tar.bz2', 'squad_bn.py']


In [20]:
from huggingface_hub import hf_hub_download

archive_path = hf_hub_download(
    repo_id="csebuetnlp/squad_bn",
    filename="data/squad_bn.tar.bz2",
    repo_type="dataset"
)
print(archive_path)

data/squad_bn.tar.bz2: reconstructing file:   0%|          |  0.00B / 8.43MB            

data/squad_bn.tar.bz2: downloading bytes:           |  0.00B            

/root/.cache/huggingface/hub/datasets--csebuetnlp--squad_bn/snapshots/1dcbca66e65528e8d0bfb69eb32230d693a80630/data/squad_bn.tar.bz2


In [21]:
import tarfile

with tarfile.open(archive_path, "r:bz2") as tar:
    tar.extractall("squad_bn_extracted")

import os
for root, dirs, files in os.walk("squad_bn_extracted"):
    for f in files:
        print(os.path.join(root, f))

/tmp/ipykernel_1240/1981117537.py:4: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extractall("squad_bn_extracted")


squad_bn_extracted/squad_bn/test.json
squad_bn_extracted/squad_bn/validation.json
squad_bn_extracted/squad_bn/train.json


In [22]:
import json

with open("squad_bn_extracted/squad_bn/train.json", encoding="utf-8") as f:
    train_raw = json.load(f)

# inspect the top-level structure
print(type(train_raw))
print(train_raw.keys() if isinstance(train_raw, dict) else "not a dict")

<class 'dict'>
dict_keys(['data'])


In [23]:
# peek one level deeper
if "data" in train_raw:
    print(len(train_raw["data"]), "articles")
    print(json.dumps(train_raw["data"][0], ensure_ascii=False, indent=2)[:1500])
else:
    print(json.dumps(train_raw, ensure_ascii=False, indent=2)[:1500])

477 articles
{
  "title": "Normans",
  "paragraphs": [
    {
      "qas": [
        {
          "question": "কোন দেশে নরম্যান্ডি অবস্থিত?",
          "id": "56ddde6b9a695914005b9628",
          "answers": [
            {
              "text": "ফ্রান্স",
              "answer_start": 103
            },
            {
              "text": "ফ্রান্স",
              "answer_start": 103
            },
            {
              "text": "ফ্রান্স",
              "answer_start": 103
            },
            {
              "text": "ফ্রান্স",
              "answer_start": 103
            }
          ]
        },
        {
          "question": "নরমান্ডিতে নরম্যানরা কখন ছিল?",
          "id": "56ddde6b9a695914005b9629",
          "answers": [
            {
              "text": "দশম এবং একাদশ শতাব্দীতে",
              "answer_start": 79
            },
            {
              "text": "দশম এবং একাদশ শতাব্দীতে",
              "answer_start": 79
            },
            {
              "text

In [24]:
import json

with open("squad_bn_extracted/squad_bn/train.json", encoding="utf-8") as f:
    train_raw = json.load(f)

print(type(train_raw))
print(train_raw.keys() if isinstance(train_raw, dict) else "not a dict")

<class 'dict'>
dict_keys(['data'])


In [25]:
if "data" in train_raw:
    print(len(train_raw["data"]), "articles")
    print(json.dumps(train_raw["data"][0], ensure_ascii=False, indent=2)[:1500])
else:
    print(json.dumps(train_raw, ensure_ascii=False, indent=2)[:1500])

477 articles
{
  "title": "Normans",
  "paragraphs": [
    {
      "qas": [
        {
          "question": "কোন দেশে নরম্যান্ডি অবস্থিত?",
          "id": "56ddde6b9a695914005b9628",
          "answers": [
            {
              "text": "ফ্রান্স",
              "answer_start": 103
            },
            {
              "text": "ফ্রান্স",
              "answer_start": 103
            },
            {
              "text": "ফ্রান্স",
              "answer_start": 103
            },
            {
              "text": "ফ্রান্স",
              "answer_start": 103
            }
          ]
        },
        {
          "question": "নরমান্ডিতে নরম্যানরা কখন ছিল?",
          "id": "56ddde6b9a695914005b9629",
          "answers": [
            {
              "text": "দশম এবং একাদশ শতাব্দীতে",
              "answer_start": 79
            },
            {
              "text": "দশম এবং একাদশ শতাব্দীতে",
              "answer_start": 79
            },
            {
              "text

In [26]:
def flatten_squad(raw_json):
    rows = []
    for article in raw_json["data"]:
        for paragraph in article["paragraphs"]:
            context = paragraph["context"]
            for qa in paragraph["qas"]:
                if len(qa["answers"]) == 0:
                    continue  # skip unanswerable questions for now — simpler for a first pass
                # dedupe identical (text, answer_start) pairs, keep first occurrence order
                seen = set()
                texts, starts = [], []
                for ans in qa["answers"]:
                    key = (ans["text"], ans["answer_start"])
                    if key not in seen:
                        seen.add(key)
                        texts.append(ans["text"])
                        starts.append(ans["answer_start"])
                rows.append({
                    "id": qa["id"],
                    "title": article["title"],
                    "context": context,
                    "question": qa["question"],
                    "answers": {"text": texts, "answer_start": starts},
                })
    return rows

train_rows = flatten_squad(train_raw)
print(len(train_rows), "training examples")
print(train_rows[0])

68674 training examples
{'id': '56ddde6b9a695914005b9628', 'title': 'Normans', 'context': 'নর্মানরা (নর্মান: নুরমান্দ; ফরাসি: নরমান্ড; লাতিন: নরমান্নি) ছিল সেই জাতি যারা দশম এবং একাদশ শতাব্দীতে ফ্রান্সের একটি অঞ্চল নরমান্ডিতে তাদের নাম দিয়েছিল। তারা নর্স ("নরম্যান" এসেছে ডেনমার্ক, আইসল্যান্ড এবং নরওয়ে থেকে আগত "নর্সম্যান") হানাদার এবং জলদস্যুদের থেকে, যারা তাদের নেতা রোলোর অধীনে পশ্চিম ফ্রান্সিয়ার রাজা তৃতীয় চার্লসের কাছে আনুগত্যের শপথ নিতে সম্মত হয়েছিল। প্রজন্মের পর প্রজন্ম ধরে স্থানীয় ফ্রাঙ্কিশ এবং রোমান-গৌলিশ জনগোষ্ঠীর সাথে মিশে যাওয়ার মাধ্যমে, তাদের বংশধররা ধীরে ধীরে পশ্চিম ফ্রান্সিয়ার ক্যারোলিনজিয়ান-ভিত্তিক সংস্কৃতির সাথে মিশে যাবে। নর্মানদের স্বতন্ত্র সাংস্কৃতিক ও জাতিগত পরিচয় প্রথম দিকে দশম শতাব্দীর প্রথমার্ধে উদ্ভূত হয়েছিল এবং পরবর্তী শতাব্দীগুলিতে এটি বিবর্তিত হতে থাকে।', 'question': 'কোন দেশে নরম্যান্ডি অবস্থিত?', 'answers': {'text': ['ফ্রান্স'], 'answer_start': [103]}}


In [27]:
with open("squad_bn_extracted/squad_bn/validation.json", encoding="utf-8") as f:
    val_raw = json.load(f)
with open("squad_bn_extracted/squad_bn/test.json", encoding="utf-8") as f:
    test_raw = json.load(f)

val_rows = flatten_squad(val_raw)
test_rows = flatten_squad(test_raw)

print(len(val_rows), "validation examples")
print(len(test_rows), "test examples")

1251 validation examples
1252 test examples


In [28]:
from datasets import Dataset, DatasetDict

raw_dataset = DatasetDict({
    "train": Dataset.from_list(train_rows),
    "validation": Dataset.from_list(val_rows),
    "test": Dataset.from_list(test_rows),
})

print(raw_dataset)
print(raw_dataset["train"][0])

DatasetDict({
    train: Dataset({
        features: ['id', 'title', 'context', 'question', 'answers'],
        num_rows: 68674
    })
    validation: Dataset({
        features: ['id', 'title', 'context', 'question', 'answers'],
        num_rows: 1251
    })
    test: Dataset({
        features: ['id', 'title', 'context', 'question', 'answers'],
        num_rows: 1252
    })
})
{'id': '56ddde6b9a695914005b9628', 'title': 'Normans', 'context': 'নর্মানরা (নর্মান: নুরমান্দ; ফরাসি: নরমান্ড; লাতিন: নরমান্নি) ছিল সেই জাতি যারা দশম এবং একাদশ শতাব্দীতে ফ্রান্সের একটি অঞ্চল নরমান্ডিতে তাদের নাম দিয়েছিল। তারা নর্স ("নরম্যান" এসেছে ডেনমার্ক, আইসল্যান্ড এবং নরওয়ে থেকে আগত "নর্সম্যান") হানাদার এবং জলদস্যুদের থেকে, যারা তাদের নেতা রোলোর অধীনে পশ্চিম ফ্রান্সিয়ার রাজা তৃতীয় চার্লসের কাছে আনুগত্যের শপথ নিতে সম্মত হয়েছিল। প্রজন্মের পর প্রজন্ম ধরে স্থানীয় ফ্রাঙ্কিশ এবং রোমান-গৌলিশ জনগোষ্ঠীর সাথে মিশে যাওয়ার মাধ্যমে, তাদের বংশধররা ধীরে ধীরে পশ্চিম ফ্রান্সিয়ার ক্যারোলিনজিয়ান-ভিত্তিক সংস্কৃতির সাথ

In [29]:
from transformers import AutoTokenizer, AutoModelForQuestionAnswering

model_checkpoint = "csebuetnlp/banglabert"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)
model = AutoModelForQuestionAnswering.from_pretrained(model_checkpoint)

config.json:   0%|          | 0.00/586 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/119 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/528k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  443MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] ElectraForQuestionAnswering LOAD REPORT from: csebuetnlp/banglabert
Key                                               | Status     | 
--------------------------------------------------+------------+-
discriminator_predictions.dense_prediction.bias   | UNEXPECTED | 
discriminator_predictions.dense.weight            | UNEXPECTED | 
discriminator_predictions.dense_prediction.weight | UNEXPECTED | 
discriminator_predictions.dense.bias              | UNEXPECTED | 
qa_outputs.bias                                   | MISSING    | 
qa_outputs.weight                                 | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [30]:
max_length = 384
stride = 128  # overlap between chunks if context is long

def preprocess_training(examples):
    questions = [q.strip() for q in examples["question"]]
    inputs = tokenizer(
        questions,
        examples["context"],
        max_length=max_length,
        truncation="only_second",
        stride=stride,
        return_overflowing_tokens=True,
        return_offsets_mapping=True,
        padding="max_length",
    )

    offset_mapping = inputs.pop("offset_mapping")
    sample_map = inputs.pop("overflow_to_sample_mapping")
    answers = examples["answers"]
    start_positions = []
    end_positions = []

    for i, offsets in enumerate(offset_mapping):
        sample_idx = sample_map[i]
        answer = answers[sample_idx]
        start_char = answer["answer_start"][0]
        end_char = start_char + len(answer["text"][0])
        sequence_ids = inputs.sequence_ids(i)

        # find start/end of context in tokenized sequence
        idx = 0
        while sequence_ids[idx] != 1:
            idx += 1
        context_start = idx
        while sequence_ids[idx] == 1:
            idx += 1
        context_end = idx - 1

        # if answer not fully inside this chunk, label as (0,0) = unanswerable
        if offsets[context_start][0] > start_char or offsets[context_end][1] < end_char:
            start_positions.append(0)
            end_positions.append(0)
        else:
            idx = context_start
            while idx <= context_end and offsets[idx][0] <= start_char:
                idx += 1
            start_positions.append(idx - 1)
            idx = context_end
            while idx >= context_start and offsets[idx][1] >= end_char:
                idx -= 1
            end_positions.append(idx + 1)

    inputs["start_positions"] = start_positions
    inputs["end_positions"] = end_positions
    return inputs

train_dataset = raw_dataset["train"].map(
    preprocess_training, batched=True, remove_columns=raw_dataset["train"].column_names
)

Map:   0%|          | 0/68674 [00:00<?, ? examples/s]

In [34]:
# Process validation set the same way as train, for loss-based eval during training
eval_dataset = raw_dataset["validation"].map(
    preprocess_training, batched=True, remove_columns=raw_dataset["validation"].column_names
)
print(eval_dataset)

Map:   0%|          | 0/1251 [00:00<?, ? examples/s]

Dataset({
    features: ['input_ids', 'token_type_ids', 'attention_mask', 'start_positions', 'end_positions'],
    num_rows: 1313
})


In [35]:
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    output_dir="./banglabert-qa",
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    learning_rate=3e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=2,
    weight_decay=0.01,
    fp16=True,
    logging_steps=50,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,     # now provided
    processing_class=tokenizer,
)

trainer.train()

Epoch,Training Loss,Validation Loss
1,0.886345,1.466160
2,0.437967,1.570100


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=17348, training_loss=0.7780448033124147, metrics={'train_runtime': 2940.3676, 'train_samples_per_second': 47.196, 'train_steps_per_second': 5.9, 'total_flos': 2.719589708946125e+16, 'train_loss': 0.7780448033124147, 'epoch': 2.0})

In [40]:
import collections
import numpy as np
import evaluate
import torch

metric = evaluate.load("squad")

def preprocess_validation(examples):
    questions = [q.strip() for q in examples["question"]]
    inputs = tokenizer(
        questions, examples["context"],
        max_length=max_length, truncation="only_second",
        stride=stride, return_overflowing_tokens=True,
        return_offsets_mapping=True, padding="max_length",
    )
    sample_map = inputs.pop("overflow_to_sample_mapping")
    example_ids = []
    for i in range(len(inputs["input_ids"])):
        sample_idx = sample_map[i]
        example_ids.append(examples["id"][sample_idx])
        sequence_ids = inputs.sequence_ids(i)
        offset = inputs["offset_mapping"][i]
        inputs["offset_mapping"][i] = [
            o if sequence_ids[k] == 1 else None for k, o in enumerate(offset)
        ]
    inputs["example_id"] = example_ids
    return inputs

val_raw_slice = raw_dataset["validation"].select(range(500))
val_dataset = val_raw_slice.map(preprocess_validation, batched=True, remove_columns=val_raw_slice.column_names)

# ---- Prediction: manual numpy batching, avoids the broken torchvision import ----
eval_set_for_model = val_dataset.remove_columns(["example_id", "offset_mapping"])

input_ids = np.array(eval_set_for_model["input_ids"])
attention_mask = np.array(eval_set_for_model["attention_mask"])
token_type_ids = np.array(eval_set_for_model["token_type_ids"]) if "token_type_ids" in eval_set_for_model.column_names else None

batch_size = 16
n = len(input_ids)

all_start_logits = []
all_end_logits = []

trainer.model.eval()
with torch.no_grad():
    for i in range(0, n, batch_size):
        batch_input_ids = torch.tensor(input_ids[i:i+batch_size]).to(trainer.model.device)
        batch_attention_mask = torch.tensor(attention_mask[i:i+batch_size]).to(trainer.model.device)
        inputs = {"input_ids": batch_input_ids, "attention_mask": batch_attention_mask}
        if token_type_ids is not None:
            inputs["token_type_ids"] = torch.tensor(token_type_ids[i:i+batch_size]).to(trainer.model.device)

        outputs = trainer.model(**inputs)
        all_start_logits.append(outputs.start_logits.cpu().numpy())
        all_end_logits.append(outputs.end_logits.cpu().numpy())

start_logits = np.concatenate(all_start_logits, axis=0)
end_logits = np.concatenate(all_end_logits, axis=0)
print("Logits shape:", start_logits.shape, end_logits.shape)
# ---- end prediction block ----

example_to_features = collections.defaultdict(list)
for idx, feature_id in enumerate(val_dataset["example_id"]):
    example_to_features[feature_id].append(idx)

n_best = 20
max_answer_length = 30
predicted_answers = []

for example in val_raw_slice:
    example_id = example["id"]
    context = example["context"]
    answers = []
    for feature_index in example_to_features[example_id]:
        start_logit = start_logits[feature_index]
        end_logit = end_logits[feature_index]
        offsets = val_dataset[feature_index]["offset_mapping"]

        start_indexes = np.argsort(start_logit)[-n_best:][::-1]
        end_indexes = np.argsort(end_logit)[-n_best:][::-1]
        for start_index in start_indexes:
            for end_index in end_indexes:
                if offsets[start_index] is None or offsets[end_index] is None:
                    continue
                if end_index < start_index or end_index - start_index + 1 > max_answer_length:
                    continue
                answers.append({
                    "text": context[offsets[start_index][0]: offsets[end_index][1]],
                    "logit_score": start_logit[start_index] + end_logit[end_index],
                })
    if answers:
        best_answer = max(answers, key=lambda x: x["logit_score"])
        predicted_answers.append({"id": example_id, "prediction_text": best_answer["text"]})
    else:
        predicted_answers.append({"id": example_id, "prediction_text": ""})

theoretical_answers = [{"id": ex["id"], "answers": ex["answers"]} for ex in val_raw_slice]

results = metric.compute(predictions=predicted_answers, references=theoretical_answers)
print(results)

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

Logits shape: (525, 384) (525, 384)
{'exact_match': 57.2, 'f1': 70.90499271798036}


In [41]:
trainer.save_model("banglabert-qa-final")
tokenizer.save_pretrained("banglabert-qa-final")

# zip it up for easy download
!zip -r banglabert-qa-final.zip banglabert-qa-final

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

  adding: banglabert-qa-final/ (stored 0%)
  adding: banglabert-qa-final/tokenizer.json (deflated 76%)
  adding: banglabert-qa-final/config.json (deflated 55%)
  adding: banglabert-qa-final/tokenizer_config.json (deflated 46%)
  adding: banglabert-qa-final/training_args.bin (deflated 53%)
  adding: banglabert-qa-final/model.safetensors (deflated 7%)
